# Product mapping update 2026-11-15: other glucose drugs in the patient journey

**Why.** Open point 5 / decision X5: the patient journey had the other glucose drugs (sulfonylureas, pioglitazone, acarbose and others) switched off, so NB04 could not build its 110 "other glucose mgmt" stage x drug columns (7 of the 277 model claims features). In 3.0 these drugs were in the class "Other- glucose mgmt" of the Snowflake view. The user decided on 2026-11-15 to switch them on.

**What changes (Rx basket only).** Input `Kerendia_Rx_Final_Product_Mapping_2026_11_13.csv`, output `Kerendia_Rx_Final_Product_Mapping_2026_11_15.csv` (3.1 volume, `final_product_mapping/`). The 2026_11_13 file is not touched. Px / Dx / Lab baskets are unchanged (the config picks the latest date per type).

| Change | Rows | Values |
| --- | --- | --- |
| Switch on the rows with the note "Not in the 3.0 journey: view category 'Other- glucose mgmt'" | 18 (for example GLIPIZIDE, GLIMEPIRIDE, PIOGLITAZONE HYDROCHLORID, ACARBOSE, SYMLINPEN 60) | `ACTIVE_JOURNEY = Y`, `TREATMENT_CATEGORY = Other- glucose mgmt`, `BRAND_LABEL` empty (no brand of its own, like metformin) |
| Not changed | Products in the HCP Rx pull that are not in the basket (GLUCOTROL XL, BYDUREON PEN, AMARYL, GLUCOTROL, PRANDIN, AVANDIA, ACTOPLUS MET XR, TOLBUTAMIDE: 20,064 claims 2023-01 to 2026-09) | Open question, not part of this file |

**Code change that goes with this file** (same date): NB04 stage x drug step maps GLP-1 and DPP-IV to the class "Other- glucose mgmt" (as 3.0 and as NB01) and keeps each patient-month, class and brand once.

In [0]:
import pandas as pd, hashlib, os

FPM = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/final_product_mapping/"
OLD_DATE, NEW_DATE = "2026_11_13", "2026_11_15"
SRC = FPM + f"Kerendia_Rx_Final_Product_Mapping_{OLD_DATE}.csv"
DST = FPM + f"Kerendia_Rx_Final_Product_Mapping_{NEW_DATE}.csv"
CHANGE = f"3.1 change {NEW_DATE.replace('_', '-')}"
WHY = "other glucose drugs in the patient journey (open point 5, decision X5 revised 2026-11-15; 3.0 view category 'Other- glucose mgmt')"
OG = "Other- glucose mgmt"
OG_NOTE = "Not in the 3.0 journey: view category 'Other- glucose mgmt'"

# The 18 inactive rows that the 3.0 view put in 'Other- glucose mgmt'
OG_SWITCH_ON = ["GLIPIZIDE", "GLIMEPIRIDE", "GLIPIZIDE ER", "PIOGLITAZONE HYDROCHLORID", "GLYBURIDE", "PIOGLITAZONE HCL",
                "ACARBOSE", "GLIPIZIDE/METFORMIN HYDRO", "REPAGLINIDE", "GLYBURIDE/METFORMIN HYDRO", "NATEGLINIDE",
                "PIOGLITAZONE HCL/METFORMI", "SYMLINPEN 60", "PIOGLITAZONE HCL-GLIMEPIR", "GLYBURIDE MICRONIZED",
                "MIGLITOL", "SYMLINPEN 120", "GLIPIZIDE XL"]

md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
src_md5 = md5(SRC)
assert not os.path.exists(DST), f"{DST} already exists"

rx_old = pd.read_csv(SRC, dtype=str, keep_default_na=False)
rx_new = rx_old.copy()

# The list above must be exactly the inactive rows with the 3.0 'Other- glucose mgmt' note
_noted = rx_old[(rx_old.ACTIVE_JOURNEY == "N") & rx_old.NOTE.str.contains(OG_NOTE, regex=False)]
assert sorted(_noted.MATCH_VALUE) == sorted(OG_SWITCH_ON), "the 18-row list does not match the basket notes"

def add_note(old, new):
    return f"{old}; {CHANGE}: {new}" if old else f"{CHANGE}: {new}"

for name in OG_SWITCH_ON:
    i = rx_new.index[(rx_new.MATCH_TYPE == "NAME") & (rx_new.MATCH_VALUE == name) & (rx_new.STRENGTH == "")]
    assert len(i) == 1, f"expected one row for {name}, found {len(i)}"
    r = rx_new.loc[i[0]]
    assert r.ACTIVE_JOURNEY == "N" and r.TREATMENT_CATEGORY == "" and r.BRAND_LABEL == "", f"{name} is not an inactive row"
    rx_new.loc[i[0], ["ACTIVE_JOURNEY", "TREATMENT_CATEGORY"]] = ["Y", OG]
    rx_new.loc[i[0], "NOTE"] = add_note(r.NOTE, f"N -> Y, {OG}. {WHY}")

# ---- Checks before writing ----
assert len(rx_old) == 372 and len(rx_new) == 372, (len(rx_old), len(rx_new))
assert (rx_old.ACTIVE_JOURNEY == "Y").sum() == 266 and (rx_new.ACTIVE_JOURNEY == "Y").sum() == 284
assert (rx_new.TREATMENT_CATEGORY == OG).sum() == 18 and (rx_new[rx_new.TREATMENT_CATEGORY == OG].BRAND_LABEL == "").all()
changed = [i for i in range(len(rx_old)) if not rx_old.iloc[i].equals(rx_new.iloc[i])]
assert sorted(rx_new.MATCH_VALUE.iloc[changed]) == sorted(OG_SWITCH_ON), "rows other than the 18 rows changed"
assert not rx_new[rx_new.STRENGTH == ""].MATCH_VALUE.duplicated().any(), "product name on more than one row"

rx_new.to_csv(DST, index=False)
assert md5(SRC) == src_md5, "the 2026_11_13 file changed"
back = pd.read_csv(DST, dtype=str, keep_default_na=False)
assert back.equals(rx_new), "written file differs from the table"
print(f"Written {DST}: {len(back)} rows, {(back.ACTIVE_JOURNEY == 'Y').sum()} active, "
      f"{(back.TREATMENT_CATEGORY == OG).sum()} '{OG}'; changed rows {len(changed)}")
display(back[back.TREATMENT_CATEGORY == OG])